In [6]:
import torch
import numpy as np
import pandas as pd
from torchvision import models, transforms
from PIL import Image
from io import BytesIO
from tqdm import tqdm
import requests
import os
device = "cuda" if torch.cuda.is_available() else "cpu"

model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
model.fc = torch.nn.Identity()
model = model.to(device)
model.eval()

print("Using device:", device)


Using device: cpu


In [7]:
#creating poster urls
def get_tmdb_poster_url(poster_path, size="w500"):
    if poster_path is None or str(poster_path).strip() == "":
        return None
    return f"https://image.tmdb.org/t/p/{size}{poster_path}"


In [8]:
movies=pd.read_csv("D:\MINI PROJECT\DATASET\movies_final.csv")

In [9]:
#poster extraction
movies = pd.read_csv(r"D:\MINI PROJECT\DATASET\movies_final.csv")
num_movies = len(movies)
print("Total movies:", num_movies)


Total movies: 23138


In [10]:
#preprocessing and embedding functions
preprocess = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

def extract_poster_embedding(image):
    image_tensor = preprocess(image).unsqueeze(0).to(device)

    with torch.no_grad():
        features = model(image_tensor)

    return features.squeeze().cpu().numpy()


In [11]:
#resume safe initialization
EMBED_PATH = "poster_embeddings.npy"
STATUS_PATH = "poster_status.npy"

EMB_DIM = 2048

if os.path.exists(EMBED_PATH) and os.path.exists(STATUS_PATH):
    poster_embeddings = np.load(EMBED_PATH)
    status = np.load(STATUS_PATH)
    print("Resuming extraction")
else:
    poster_embeddings = np.zeros((num_movies, EMB_DIM), dtype=np.float32)
    status = np.zeros(num_movies, dtype=np.int8)  # 0 = not done, 1 = success, -1 = failed
    print("Starting fresh")

print("Already done:", np.sum(status == 1))
print("Already failed:", np.sum(status == -1))


Resuming extraction
Already done: 9056
Already failed: 14082


In [12]:
EMBED_PATH = r"D:\MINI PROJECT\NOTEBOOKS\poster_embeddings.npy"
STATUS_PATH = r"D:\MINI PROJECT\NOTEBOOKS\poster_status.npy"
num_movies = len(movies)
EMB_DIM = 2048

if os.path.exists(EMBED_PATH) and os.path.exists(STATUS_PATH):
    poster_embeddings = np.load(EMBED_PATH)
    status = np.load(STATUS_PATH)
    print("Resuming extraction")
else:
    poster_embeddings = np.zeros((num_movies, EMB_DIM), dtype=np.float32)
    status = np.zeros(num_movies, dtype=np.int8)
    print("Starting fresh")

print("Already extracted:", np.sum(status == 1))
print("Already failed:", np.sum(status == -1))


Resuming extraction
Already extracted: 9056
Already failed: 14082


In [13]:
failed = int((status == -1).sum())

for i in tqdm(range(num_movies), desc="Extracting poster embeddings"):
    if status[i] != 0:
        continue

    try:
        poster_path = movies.loc[i, "poster_path"]
        url = get_tmdb_poster_url(poster_path)

        if url is None:
            raise ValueError("No poster")

        response = requests.get(url, timeout=10)
        response.raise_for_status()

        image = Image.open(BytesIO(response.content)).convert("RGB")
        emb = extract_poster_embedding(image)

        poster_embeddings[i] = emb
        status[i] = 1

    except Exception:
        status[i] = -1
        failed += 1

    if i % 100 == 0:
        np.save(EMBED_PATH, poster_embeddings)
        np.save(STATUS_PATH, status)

np.save(EMBED_PATH, poster_embeddings)
np.save(STATUS_PATH, status)

print("Done.")
print("Failed:", failed)


Extracting poster embeddings: 100%|██████████| 23138/23138 [00:00<00:00, 5842030.22it/s]


Done.
Failed: 14082


In [15]:
norms = np.linalg.norm(poster_embeddings, axis=1)

print("Min norm:", norms.min())
print("Mean norm:", norms.mean())
print("Zero vectors:", np.sum(norms == 0))
print("Successful embeddings:", np.sum(status == 1))


Min norm: 0.0
Mean norm: 5.784865
Zero vectors: 14082
Successful embeddings: 9056
